In [35]:
!pip install gTTS --no-deps
!pip install pyngrok fastapi uvicorn nest-asyncio

In [36]:
from gtts import gTTS
from IPython.display import Audio, display

def text_to_speech(text, lang='en', filename="output.mp3"):
    print(f"Generating audio for language: '{lang}'...")
    tts = gTTS(text=text, lang=lang, slow=False)
    tts.save(filename)
    print(f" Saved as {filename}. Playing audio below:")
    display(Audio(filename, autoplay=True))

# --- TEST 1: English ---
eng_text = "Hello! Your speech-to-text code is working perfectly. Now we are testing text to speech."
text_to_speech(eng_text, lang='en', filename="english_test.mp3")

# --- TEST 2: Hindi ---
hindi_text = "नमस्ते! आपका टेक्स्ट टू स्पीच सिस्टम पूरी तरह से काम कर रहा है।"
text_to_speech(hindi_text, lang='hi', filename="hindi_test.mp3")

Generating audio for language: 'en'...
 Saved as english_test.mp3. Playing audio below:


Generating audio for language: 'hi'...
 Saved as hindi_test.mp3. Playing audio below:


In [2]:
import os
import uuid
from fastapi import FastAPI, HTTPException
from fastapi.responses import FileResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from gtts import gTTS

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class TTSRequest(BaseModel):
    text: str
    language: str = "en"

@app.get("/")
def read_root():
    return {"message": "TTS Service is live and running!"}

@app.post("/text-to-speech")
async def text_to_speech_endpoint(data: TTSRequest):
    if not data.text.strip():
        raise HTTPException(status_code=400, detail="Text content cannot be empty.")

    temp_filename = f"tts_{uuid.uuid4()}.mp3"

    try:
        tts = gTTS(text=data.text, lang=data.language, slow=False)
        tts.save(temp_filename)

        return FileResponse(
            path=temp_filename,
            media_type="audio/mpeg",
            filename="speech.mp3"
        )
    except Exception as e:
        if os.path.exists(temp_filename):
            os.remove(temp_filename)
        raise HTTPException(status_code=500, detail=str(e))

In [3]:
import asyncio
import nest_asyncio
import uvicorn
from pyngrok import ngrok
from tts_service import app

# 1. Provide your authtoken to pyngrok so it can authenticate your session
NGROK_TOKEN = "3Ff1BERH5y1ApUfytKIzYacu5ke_ns5oJctGpiasmubg1pRz"
ngrok.set_auth_token(NGROK_TOKEN)

# 2. Allow async loops to nest safely in Colab
nest_asyncio.apply()

# 3. Clear out any old lingering tunnels
ngrok.kill()

# 4. Open a fresh tunnel on port 8002
public_url = ngrok.connect(8002)
print("\n" + "="*50)
print(f"PUBLIC TTS API URL: {public_url}")
print("==================================================\n")

# 5. Configure and spin up the server task
config = uvicorn.Config(app, host="0.0.0.0", port=8002, log_level="info")
server = uvicorn.Server(config)

loop = asyncio.get_event_loop()
loop.create_task(server.serve())


PUBLIC TTS API URL: NgrokTunnel: "https://succulent-spotting-imbecile.ngrok-free.dev" -> "http://localhost:8002"



<Task pending name='Task-1' coro=<Server.serve() running at /usr/local/lib/python3.12/dist-packages/uvicorn/server.py:77>>

In [4]:
import httpx
from IPython.display import Audio, display

#@title 🎙️ Dynamic Text-To-Speech Interface { run: "auto" }
#@markdown Type your text below and select the language, then run this cell.

# These form variables update automatically when you change them in the UI panel on the right
text_input = "Namaste, aap kaise hain" #@param {type:"string"}
language_code = "hi" #@param ["en", "hi"]

async def test_tts_api(text, lang):
    api_url = "http://localhost:8002/text-to-speech"
    payload = {
        "text": text,
        "language": lang
    }

    if not text.strip():
        print("Please enter some text first!")
        return

    print(f"Sending text to server: '{text}' [{lang}]")

    try:
        async with httpx.AsyncClient(timeout=20.0) as client:
            response = await client.post(api_url, json=payload)

        if response.status_code == 200:
            output_file = f"api_received_{lang}.mp3"
            with open(output_file, "wb") as f:
                f.write(response.content)
            print("Audio generated successfully!")
            display(Audio(output_file, autoplay=True))
        else:
            print(f"Error from server: {response.status_code} - {response.text}")

    except Exception as e:
        print(f"❌ Connection Error: Is your Cell 4 server running? ({str(e)})")

# Execute the dynamic inputs
await test_tts_api(text_input, language_code)

Sending text to server: 'Namaste, aap kaise hain' [hi]
INFO:     127.0.0.1:59824 - "POST /text-to-speech HTTP/1.1" 200 OK
Audio generated successfully!
